# Task 6: Unsupervised Learning — Clustering

**Objective:** Apply K-Means Clustering on the CoreTech client dataset to group clients by similar features.

### Steps:
1. Select relevant features
2. Scale the data
3. Apply K-Means with different k values
4. Use Elbow Method to find the best k
5. Visualize clusters using scatter plot
6. Analyze each cluster
7. Apply Hierarchical Clustering and compare results
8. Write business insights about each client group

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries imported successfully!')

## 2. Load the CoreTech Client Dataset

In [ ]:
# Load the dataset
# Replace 'coretech_clients.csv' with your actual file path
df = pd.read_csv('coretech_clients.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info and statistics
print('Dataset Info:')
print(df.info())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# Check for missing values
print('Missing Values per Column:')
print(df.isnull().sum())

# Drop rows with missing values (or impute as needed)
df.dropna(inplace=True)
print('\nShape after dropping nulls:', df.shape)

## 4. Feature Selection

Select numerical features that are relevant for grouping clients by similar characteristics.

In [ ]:
# Select relevant numerical features for clustering
# Update these column names based on your actual dataset
features = ['annual_revenue', 'num_employees', 'contract_value', 'support_tickets', 'usage_score']

X = df[features].copy()

print('Selected Features:')
print(X.head())

## 5. Feature Scaling

Standardize the data so that all features contribute equally to the distance calculations in K-Means.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled Data (first 5 rows):')
print(pd.DataFrame(X_scaled, columns=features).head())

## 6. Elbow Method — Finding the Optimal Number of Clusters (k)

We iterate over different values of k and plot the Within-Cluster Sum of Squares (WCSS) to identify the "elbow" point.

In [ ]:
wcss = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, wcss, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method — WCSS vs k', fontsize=13)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS')

axes[1].plot(k_range, silhouette_scores, marker='s', color='darkorange', linewidth=2)
axes[1].set_title('Silhouette Score vs k', fontsize=13)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Elbow and Silhouette plots saved.')

## 7. Apply K-Means with Optimal k

Based on the Elbow Method and Silhouette Score, select the best k and train the final K-Means model.

In [ ]:
# Set the optimal k based on elbow/silhouette analysis
optimal_k = 3  # Update this value based on your plots

kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

df['KMeans_Cluster'] = kmeans_final.labels_

print(f'K-Means applied with k={optimal_k}')
print('\nCluster Distribution:')
print(df['KMeans_Cluster'].value_counts().sort_index())

## 8. Visualize K-Means Clusters (PCA Scatter Plot)

Reduce dimensions to 2D using PCA for visualization.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['KMeans_Cluster'] = df['KMeans_Cluster'].values

plt.figure(figsize=(9, 6))
colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261']

for cluster in range(optimal_k):
    subset = pca_df[pca_df['KMeans_Cluster'] == cluster]
    plt.scatter(subset['PC1'], subset['PC2'],
                label=f'Cluster {cluster}',
                color=colors[cluster], s=60, alpha=0.7, edgecolors='k', linewidths=0.4)

# Plot centroids
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            marker='X', s=200, c='black', label='Centroids', zorder=5)

plt.title(f'K-Means Clustering (k={optimal_k}) — PCA Projection', fontsize=13)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend()
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cluster Analysis — Profile Each Cluster

Compute the mean of each feature per cluster to understand what defines each group.

In [ ]:
cluster_profile = df.groupby('KMeans_Cluster')[features].mean().round(2)
print('Cluster Profiles (Mean Feature Values):')
cluster_profile

In [ ]:
# Heatmap of cluster profiles
plt.figure(figsize=(10, 4))
sns.heatmap(cluster_profile.T, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Mean Value'})
plt.title('Cluster Feature Profiles Heatmap', fontsize=13)
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Hierarchical Clustering

Apply Agglomerative (Hierarchical) Clustering and visualize with a dendrogram to compare with K-Means results.

In [ ]:
# Dendrogram (use a sample if the dataset is large)
sample_size = min(200, len(X_scaled))
X_sample = X_scaled[:sample_size]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode='lastp', p=30,
           leaf_rotation=45, leaf_font_size=10, show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=13)
plt.xlabel('Data Points')
plt.ylabel('Euclidean Distance')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply Agglomerative Clustering with the same k
agg_cluster = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
df['Hierarchical_Cluster'] = agg_cluster.fit_predict(X_scaled)

print('Hierarchical Cluster Distribution:')
print(df['Hierarchical_Cluster'].value_counts().sort_index())

## 11. Compare K-Means vs Hierarchical Clustering

In [ ]:
pca_df['Hierarchical_Cluster'] = df['Hierarchical_Cluster'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cluster in range(optimal_k):
    subset = pca_df[pca_df['KMeans_Cluster'] == cluster]
    axes[0].scatter(subset['PC1'], subset['PC2'],
                    label=f'Cluster {cluster}', color=colors[cluster],
                    s=50, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[0].set_title(f'K-Means (k={optimal_k})', fontsize=13)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend()

for cluster in range(optimal_k):
    subset = pca_df[pca_df['Hierarchical_Cluster'] == cluster]
    axes[1].scatter(subset['PC1'], subset['PC2'],
                    label=f'Cluster {cluster}', color=colors[cluster],
                    s=50, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[1].set_title(f'Hierarchical Clustering (k={optimal_k})', fontsize=13)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend()

plt.suptitle('K-Means vs Hierarchical Clustering Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Silhouette score comparison
kmeans_sil = silhouette_score(X_scaled, df['KMeans_Cluster'])
hier_sil   = silhouette_score(X_scaled, df['Hierarchical_Cluster'])

print(f'Silhouette Score — K-Means        : {kmeans_sil:.4f}')
print(f'Silhouette Score — Hierarchical   : {hier_sil:.4f}')
print(f'\nBetter algorithm: {"K-Means" if kmeans_sil > hier_sil else "Hierarchical"}')

## 12. Business Insights per Client Group

Based on the cluster profiles, we derive actionable business insights for each identified client segment.

In [ ]:
print('=' * 60)
print('BUSINESS INSIGHTS — CORETECH CLIENT SEGMENTS')
print('=' * 60)

insights = {
    0: {
        'name'   : 'High-Value Enterprise Clients',
        'profile': 'High revenue, large workforce, high contract value, low support tickets.',
        'insight': ('These are CoreTech\'s most profitable clients. '
                    'Focus on dedicated account management, premium SLAs, '
                    'and upselling advanced enterprise features to maximize lifetime value.')
    },
    1: {
        'name'   : 'Growing SME Clients',
        'profile': 'Moderate revenue and employees, medium contract value, moderate usage.',
        'insight': ('These clients show growth potential. '
                    'Offer scalable pricing, onboarding support, and targeted feature training '
                    'to accelerate their adoption and expand contracts over time.')
    },
    2: {
        'name'   : 'At-Risk / Low-Engagement Clients',
        'profile': 'Low revenue, small team, low contract value, high support tickets.',
        'insight': ('These clients may be struggling to realize value. '
                    'Proactive customer success outreach, simplified onboarding, '
                    'and discounted plans could improve retention and reduce churn risk.')
    }
}

for cluster_id, info in insights.items():
    print(f'\nCluster {cluster_id}: {info["name"]}')
    print(f'  Profile : {info["profile"]}')
    print(f'  Insight : {info["insight"]}')

print('\n' + '=' * 60)

## 13. Save Results

In [ ]:
# Save clustered dataset
df.to_csv('coretech_clients_clustered.csv', index=False)
print('Clustered dataset saved to: coretech_clients_clustered.csv')
print('\nSaved plots:')
print('  - elbow_silhouette.png')
print('  - kmeans_clusters.png')
print('  - cluster_heatmap.png')
print('  - dendrogram.png')
print('  - comparison_clusters.png')

## Summary

| Step | Method | Outcome |
|------|--------|---------|
| Feature Selection | Domain knowledge | Numerical client features selected |
| Scaling | StandardScaler | Mean=0, Std=1 normalized features |
| Optimal k | Elbow + Silhouette | Best k identified |
| Clustering | K-Means (k-means++) | Clients grouped into k clusters |
| Comparison | Agglomerative (Ward) | Hierarchical clustering compared |
| Insights | Cluster profiling | Business strategies per segment derived |

---
*Notebook prepared for Task 6 — CoreTech Unsupervised Learning Assignment*